# Rule: **build_heat_totals**

**Description**
 
Approximate heat demand for (residential, services) x (space) x (total, electricity) based on:

- energy totals, from `build_energy_totals`
- heating degree days (HDD). Historical data from `COUNTRY_HDD_DATASET['folder']/era5-HDD-per-country.csv`

A linear regression of heat demand on HDDs is performed from 2007-2021 data. Demands are estimated for other years (especially 2022, 2023) from historial HDDs data.

**NOTE: This seems to introduce outliers for year 2022**.

**Outputs**

- resources/`heat_totals.csv`

In [ ]:
######################################## Parameters

### Run
name = 'case_SectorCoupled'
prefix = ''

In [ ]:
##### Import packages
import matplotlib.pyplot as plt
import os 
import sys
import numpy as np


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `heat_totals.csv`

Load the file and preview its content.

In [ ]:
file = "heat_totals.csv"

df_heat_totals = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_heat_totals.head()

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### Side-by-side bar charts: residential and services space (total vs electricity)
required_cols = [
    "country",
    "year",
    "total residential space",
    "electricity residential space",
    "total services space",
    "electricity services space",
]

missing_cols = [c for c in required_cols if c not in df_heat_totals.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

df_country = df_heat_totals[df_heat_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

plot_df = (
    df_country.groupby("year", dropna=False)[
        [
            "total residential space",
            "electricity residential space",
            "total services space",
            "electricity services space",
        ]
    ]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

ymax = max(plot_df.max()) * 1.05

fig, (ax_res, ax_srv) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Residential
ax_res.bar(
    x - center_shift,
    plot_df["total residential space"].values,
    width=width,
    color="#4E79A7",
    alpha=0.8,
    label="Total",
)
ax_res.bar(
    x + center_shift,
    plot_df["electricity residential space"].values,
    width=width,
    color="#F28E2B",
    alpha=0.8,
    label="Electricity",
)
ax_res.set_title("Residential space")
ax_res.set_xlabel("Year")
ax_res.set_ylabel("Value")
ax_res.set_xticks(x)
ax_res.set_xticklabels(plot_df.index.astype(str), rotation=45)
ax_res.set_ylim(0, ymax)
ax_res.grid(axis="y", alpha=0.25)
ax_res.legend()

# Services
ax_srv.bar(
    x - center_shift,
    plot_df["total services space"].values,
    width=width,
    color="#4E79A7",
    alpha=0.8,
    label="Total",
)
ax_srv.bar(
    x + center_shift,
    plot_df["electricity services space"].values,
    width=width,
    color="#F28E2B",
    alpha=0.8,
    label="Electricity",
)
ax_srv.set_title("Services space")
ax_srv.set_xlabel("Year")
ax_srv.set_ylabel("Value")
ax_srv.set_xticks(x)
ax_srv.set_xticklabels(plot_df.index.astype(str), rotation=45)
ax_srv.set_ylim(0, ymax)
ax_srv.grid(axis="y", alpha=0.25)
ax_srv.legend()

fig.suptitle(f"{country} space heat demand: total vs electricity", y=1.02, fontsize=14)
plt.tight_layout()